In [ ]:
#importing necessary modules
import os
import numpy as np
import pandas as pd
import sys
print(sys.version)
print("numpy", np.__version__)
print("pandas", pd.__version__)

In [ ]:
path = "/path/to/lab_storage/genome_annotations/mouse/GENCODE.VM33.knownGene_UCSCTableBrowser_downl09.04.2024.csv"
allGenecode = pd.read_csv(path, sep = ",")
allGenecode = allGenecode.loc[:, ["name", "chrom", "strand", "chromStart", "chromEnd"]]
allGenecode

In [ ]:
chromosomes = ['chr' + str(i) for i in np.arange(1, 20, 1)] + ["chrX", "chrY"]
chromosomes

In [ ]:
chromosomes_name_table = "/path/to/lab_storage/genomes/mouse/Chromosome_naming_table_17.09.2024_Vakil.tsv"
chromosomes_name_table = pd.read_csv(chromosomes_name_table, sep = "\t", header = 0)
chromosomes_name_table = chromosomes_name_table[chromosomes_name_table["UCSC style name"].isin(chromosomes)]
chromosomes_name_table = chromosomes_name_table.loc[:, ["RefSeq seq accession", "UCSC style name"]]
chromosomes_name_table = chromosomes_name_table.rename(columns = {"UCSC style name" : "chrom"})
chromosomes_name_table

In [ ]:
path = "/path/to/lab_storage/genome_annotations/mouse/GENCODE.VM33.knownCanonical_UCSCTableBrowser_downl09.04.2024.csv"
canonGenecode = pd.read_csv(path, sep = ",")
canonGenecode["Gene ID"] = canonGenecode["protein"].str.split(".").str[0]
#
canonGenecode = canonGenecode[canonGenecode["chrom"].isin(chromosomes)]
#
canonGenecode = canonGenecode.loc[:, ["transcript", "Gene ID"]]
canonGenecode = canonGenecode.rename(columns = {"transcript" : "name"})
print(canonGenecode.shape)

canonGenecode = pd.merge(canonGenecode, allGenecode, on = "name", how = "left")
canonGenecode = pd.merge(chromosomes_name_table, canonGenecode, on = "chrom")
canonGenecode = canonGenecode.loc[:, ["RefSeq seq accession", "chromStart", "chromEnd", "Gene ID", "name", "strand"]]
canonGenecode = canonGenecode.rename(columns = {"RefSeq seq accession" : "chrom",
                                                 "chromStart" : "txStart",
                                                 "chromEnd" : "txEnd",
                                                 "name" : "#name"})

canonGenecode

In [ ]:
canonGenecode["#name"].value_counts()

In [ ]:
canonGenecode["Gene ID"].value_counts() 

In [ ]:
P_drive_path = "/path/to/lab_storage/Mouse_brain_Sept2024/gene_annotation/"

canonGenecode.to_csv(P_drive_path + "knownGenes_canononical.bed", sep = "\t", index = False, header = False)


In [ ]:
P_drive_path = "/path/to/lab_storage/Mouse_brain_Sept2024/gene_annotation/"

beyond = 2000

#upstream TSS region
DF_upstream_bounds = pd.DataFrame({})
for strand in ["-", "+"]:
    tmp = canonGenecode[canonGenecode["strand"] == strand].copy()
    if strand == "+":
        tmp.loc[:, "End"] = tmp["txStart"]
        tmp.loc[:, "Start"] = tmp["txStart"] - beyond
    if strand == "-":
        tmp.loc[:, "Start"] = tmp["txEnd"]
        tmp.loc[:, "End"] = tmp["txEnd"] + beyond
        
    tmp = tmp.loc[:, ["chrom", "Start", "End", "Gene ID", "#name", "strand"]]
    DF_upstream_bounds = pd.concat([DF_upstream_bounds, tmp])
    
DF_upstream_bounds.to_csv(P_drive_path + "knownGenes_canonTr_"+str(beyond)+"bp_upstream_TSS_feature.bed", 
                          header = False, index = False, sep = '\t')